# sum(CNAREA) != CAREA
 - Need to use CAREA

In [6]:
from pathlib import Path
import numpy as np
import pandas as pd

from phd_helpers.AbaqusPostprocessing import (
    inp2pv, get_field_path, get_field_df, add_field_to_mesh, parse_wallclock_time, parse_final_step_time, get_history_path
)
from phd_helpers.AbaqusPreprocessing import parse_memory_estimate

import subprocess
path_InpPipeline_main = '../../../../Computational/InpPipeline/main.py'
subprocess.run(["python", path_InpPipeline_main])

In [23]:
DIR = Path('outputs/15T3T-inp')

bone = 'tpm'
subject = '14548R'
ids = [
    ('3T', '0'), # 3 T4's across thickness
    ('15T', '1'), # 1.5 T10's across thickness
    ]
pose = 'neutral'
step = 0
frame = -1
instance = f"{bone.upper()}_INST"
metrics = ["CNAREA", "CPRESS", "U"]


data = []
for run_id_mesh, run_id in ids:
    job_name = Path(f'{run_id_mesh}-{pose}-{run_id}')

    path_inp = DIR / f'inpFiles/{subject}/inp'
    path_job = path_inp / job_name
    path_csv = path_job / 'resultCSVs'
    input_path = path_job / job_name.with_suffix('.inp')
    dat_path = path_job / job_name.with_suffix('.dat')
    sta_path = path_job / job_name.with_suffix('.sta')

    # RESULTS #
    meshes = inp2pv(input_path)
    mesh = meshes[bone]

    # Field data
    for metric in metrics:
        field_path = get_field_path(path_csv, metric, step, frame, instance)
        field_df = get_field_df(field_path)
        add_field_to_mesh(mesh, field_df)
    mesh['Umag'] = np.linalg.norm(np.column_stack((mesh['U1'], mesh['U2'], mesh['U3'])), axis=1)

    # History data
    history_data = pd.read_csv(get_history_path(path_csv, step))
    RF_data = history_data[history_data['historyOutputKey']=='RF1']
    CAREA_data = history_data[history_data['historyOutputDescription']=='Total area in contact']


    data.append({
        'job': job_name,
        'P': mesh['CPRESS'].max(),
        'CNAREA' : mesh['CNAREA'].sum(),
        'CAREA': CAREA_data['value'].iloc[frame],
        'F' : np.abs(RF_data['value'].iloc[frame]),
        'd' : parse_final_step_time(sta_path),
        'nodes' : mesh.n_points,
        'elements' : mesh.n_cells,
        'memory' : parse_memory_estimate(dat_path)['memory_to_minimize_io_gb'],
        'runtime' : parse_wallclock_time(dat_path),
    })

df = pd.DataFrame(data)

In [24]:
df

,job,P,CNAREA,CAREA,F,d,nodes,elements,memory,runtime
0,3T-neutral-0,2.657827,51.078532,51.113022,49.983337,0.389,30415,164963,3.142,715.0
1,15T-neutral-1,2.520094,59.899462,52.719173,50.041191,0.402,80624,54984,9.102,2242.0
